# Final rescue: clean additive SVC and rank blends

A linear, cache-first final pipeline. CV selection uses only the existing 25 folds. Validation is used once after CV finalists are frozen; test is used only for the two final submissions.


## 1. Data + existing cache


In [1]:
from pathlib import Path
import importlib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

import svc_multikernel_experiments as legacy
import svc_final_rescue_experiments as rescue
importlib.reload(legacy)
importlib.reload(rescue)

SEED = 0xFACED
EXISTING_CV_SEEDS = tuple(SEED + i for i in range(5))
N_SPLITS = 5
COORD_N_BINS = 8
COORD_ALPHA = 5.0
PCA_COMPONENTS = 128
SVC_N_JOBS = -1
CACHE_N_JOBS = 1
DATA_DIR = (Path('./data') if not Path('/kaggle/input').exists()
            else Path('/kaggle/input/competitions/week-03-hidden-pairs'))
CACHE_DIR = Path('.svc_multikernel_cache_v2')
RESCUE_CACHE_DIR = Path('.svc_final_rescue_cache_v1')
REBUILD_RESCUE_CACHE = False

train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df = pd.read_csv(DATA_DIR / 'validation.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
features = [c for c in train_df if c not in {'row_id', 'target'}]
assert len(features) == 512
assert features == [c for c in val_df if c not in {'row_id', 'target'}]
assert features == [c for c in test_df if c != 'row_id']
X_train = train_df[features].to_numpy()
y_train = train_df.target.to_numpy()

cache = rescue.load_existing_cv_cache(
    y_train, EXISTING_CV_SEEDS, CACHE_DIR, N_SPLITS,
    COORD_N_BINS, COORD_ALPHA, PCA_COMPONENTS)
assert cache.cache_builds == 0
assert len(cache.folds) == 25

# Only six deterministic derived views are allowed to be added to this sidecar.
rescue_cache = rescue.build_or_load_rescue_cache(
    cache, RESCUE_CACHE_DIR, REBUILD_RESCUE_CACHE, CACHE_N_JOBS)
assert len(rescue_cache.folds) == 25
print(f'legacy cache hits={cache.cache_hits}, builds={cache.cache_builds}')
print(f'rescue derived hits={rescue_cache.cache_hits}, builds={rescue_cache.cache_builds}')


legacy cache hits=25, builds=0
rescue derived hits=25, builds=0


## 2. Feature representation sanity check


In [2]:
GAMMA_MULTIPLIERS = [0.125, 0.25, 0.5, 1.0]
REPRESENTATION_C_GRID = [1.0, 3.0, 6.0, 10.0]

representation_results, representation_runs = rescue.calibrate_representations(
    cache,
    rescue_cache,
    GAMMA_MULTIPLIERS,
    REPRESENTATION_C_GRID,
    SVC_N_JOBS,
)
display(representation_results)


,representation,source_block,dim,gamma,C,quick_CV,full25_CV,full25_std,delta_vs_orig,wins_vs_orig,losses_vs_orig
0,geometry_orig,geometry,32,0.003193,10.0,0.916078,0.914880,0.011394,0.000000,0,0
1,geometry_white16,geometry,16,0.003417,10.0,0.918377,0.917002,0.011952,0.002122,18,7
2,geometry_family,geometry,14,0.014711,10.0,0.916909,0.915306,0.010947,0.000426,14,11
3,density_orig,density,14,0.005985,10.0,0.899811,0.901171,0.013576,0.000000,0,0
4,density_white5,density,5,0.026341,6.0,0.904907,0.905047,0.010986,0.003876,21,4
5,density_family,density,6,0.141479,6.0,0.906459,0.904699,0.011059,0.003528,19,6
6,coord_orig,coord,10,0.013867,3.0,0.874704,0.874701,0.013711,0.000000,0,0
7,coord_white4,coord,4,0.025821,3.0,0.879847,0.877664,0.012985,0.002963,22,3
8,aggregates_orig,aggregates,6,0.018459,1.0,0.836719,0.837635,0.016542,0.000000,0,0
9,aggregates_white4,aggregates,4,0.036865,6.0,0.839969,0.839761,0.016255,0.002126,17,8


## 3. Freeze final blocks


In [3]:
FINAL_REPRESENTATIONS = {
    'raw': 'raw_orig',
    'geometry': 'geometry_white16',
    'density': 'density_white5',
    'coord': 'coord_white4',
    'aggregates': 'aggregates_white4',
    'knn': 'knn_orig',
    'pca128': 'pca128_orig',
}
FINAL_RAW_GAMMA = 0.002496
FINAL_GAMMAS = {
    'raw': FINAL_RAW_GAMMA,
    'geometry': float(representation_runs['geometry_white16']['gamma']),
    'density': float(representation_runs['density_white5']['gamma']),
    'coord': float(representation_runs['coord_white4']['gamma']),
    'aggregates': float(representation_runs['aggregates_white4']['gamma']),
    'knn': float(representation_runs['knn_orig']['gamma']),
    'pca128': float(representation_runs['pca128_orig']['gamma']),
}

assert FINAL_REPRESENTATIONS == {
    'raw': 'raw_orig',
    'geometry': 'geometry_white16',
    'density': 'density_white5',
    'coord': 'coord_white4',
    'aggregates': 'aggregates_white4',
    'knn': 'knn_orig',
    'pca128': 'pca128_orig',
}
assert FINAL_GAMMAS['raw'] == 0.002496

final_blocks_df = pd.DataFrame([
    {
        'block': block,
        'representation': representation,
        'gamma': FINAL_GAMMAS[block],
        'standalone_C': representation_runs[representation]['C'],
        'full25_CV': representation_runs[representation]['aucs'].mean(),
    }
    for block, representation in FINAL_REPRESENTATIONS.items()
])
print('FINAL REPRESENTATIONS (FROZEN)')
print(FINAL_REPRESENTATIONS)
display(final_blocks_df)

FINAL REPRESENTATIONS (FROZEN)
{'raw': 'raw_orig', 'geometry': 'geometry_white16', 'density': 'density_white5', 'coord': 'coord_white4', 'aggregates': 'aggregates_white4', 'knn': 'knn_orig', 'pca128': 'pca128_orig'}


,block,representation,gamma,standalone_C,full25_CV
0,raw,raw_orig,0.002496,6.0,0.915123
1,geometry,geometry_white16,0.003417,10.0,0.917002
2,density,density_white5,0.026341,6.0,0.905047
3,coord,coord_white4,0.025821,3.0,0.877664
4,aggregates,aggregates_white4,0.036865,6.0,0.839761
5,knn,knn_orig,0.046973,10.0,0.882314
6,pca128,pca128_orig,0.002768,10.0,0.912761


## 4. Standalone references and fixed gammas


In [4]:
LEGACY_FINE6_C = 6.0
LEGACY_FINE6_WEIGHTS = {
    'raw': 0.68, 'geometry': 0.10, 'aggregates': 0.05,
    'density': 0.10, 'knn': 0.05, 'coord': 0.02, 'pca128': 0.00,
}
LEGACY_FINE6_GAMMAS = {
    'raw': 0.002496, 'geometry': 0.012772,
    'aggregates': 0.073837, 'density': 0.023941,
    'knn': 0.187890, 'coord': 0.055470, 'pca128': 0.001384,
}

# Mandatory historical raw guard on the unchanged cached 25 folds.
historical_raw_run = rescue.evaluate_fixed_representation(
    cache, rescue_cache, 'raw_orig', FINAL_RAW_GAMMA, 6.0, SVC_N_JOBS)
historical_raw_CV = float(historical_raw_run['aucs'].mean())
print(f'historical_raw_CV: {historical_raw_CV:.9f}')
if historical_raw_CV < 0.925:
    raise RuntimeError(
        'Historical raw sanity check failed: expected about 0.929, '
        f'got {historical_raw_CV:.9f}. Kernel search was stopped.')

# This is the untouched historical Fine6 anchor.
legacy_fine6_run = legacy.evaluate_config_grid(
    cache,
    {'mode': 'additive', 'weights': LEGACY_FINE6_WEIGHTS},
    LEGACY_FINE6_GAMMAS,
    [LEGACY_FINE6_C],
    SVC_N_JOBS,
)
reference_table = pd.DataFrame([
    {'model': 'historical_raw', 'CV_mean': historical_raw_CV,
     'CV_std': historical_raw_run['aucs'].std(),
     'C': 6.0, 'gamma': FINAL_RAW_GAMMA},
    {'model': 'legacy_fine6', 'CV_mean': legacy_fine6_run['aucs'].mean(),
     'CV_std': legacy_fine6_run['aucs'].std(),
     'C': LEGACY_FINE6_C, 'gamma': np.nan},
])
display(reference_table)

historical_raw_CV: 0.929060217


,model,CV_mean,CV_std,C,gamma
0,historical_raw,0.929060,0.007543,6.0,0.002496
1,legacy_fine6,0.932035,0.007212,6.0,NaN


## 5. Multi-kernel candidates


In [5]:
print('CORRECTED CLEAN MODEL SETTINGS')
print('FINAL_REPRESENTATIONS =', FINAL_REPRESENTATIONS)
print('FINAL_GAMMAS =', FINAL_GAMMAS)
assert FINAL_REPRESENTATIONS['geometry'] == 'geometry_white16'
assert FINAL_REPRESENTATIONS['density'] == 'density_white5'
assert FINAL_REPRESENTATIONS['coord'] == 'coord_white4'
assert FINAL_REPRESENTATIONS['aggregates'] == 'aggregates_white4'
assert FINAL_GAMMAS['raw'] == 0.002496

KERNEL_WEIGHT_CONFIGS = {
    'K0': {'raw': .68, 'geometry': .10, 'density': .10, 'aggregates': .05,
           'knn': .05, 'coord': .02, 'pca128': .00},
    'K1': {'raw': .70, 'geometry': .10, 'density': .10,
           'aggregates': .05, 'knn': .05},
    'K2': {'raw': .60, 'geometry': .18, 'density': .10, 'aggregates': .06,
           'knn': .04, 'coord': .02},
    'K3': {'raw': .60, 'geometry': .10, 'density': .18, 'aggregates': .06,
           'knn': .04, 'coord': .02},
    'K4': {'raw': .58, 'geometry': .16, 'density': .14, 'aggregates': .06,
           'knn': .04, 'coord': .02},
    'K5': {'raw': .52, 'geometry': .20, 'density': .16, 'aggregates': .06,
           'knn': .04, 'coord': .02},
    'K6': {'raw': .60, 'geometry': .14, 'density': .12, 'aggregates': .08,
           'knn': .04, 'coord': .02},
    'K7': {'raw': .56, 'geometry': .20, 'density': .16,
           'aggregates': .05, 'knn': .03},
    'K8': {'raw': .58, 'geometry': .16, 'density': .12, 'aggregates': .05,
           'knn': .03, 'coord': .01, 'pca128': .05},
    'K9': {'raw': .55, 'geometry': .16, 'density': .12, 'aggregates': .05,
           'knn': .03, 'coord': .01, 'pca128': .08},
}
KERNEL_C_GRID = [1.0, 2.0, 3.0, 6.0, 10.0, 18.0]

kernel_runs = rescue.evaluate_additive_candidates(
    cache, rescue_cache, FINAL_REPRESENTATIONS, FINAL_GAMMAS,
    KERNEL_WEIGHT_CONFIGS, KERNEL_C_GRID, SVC_N_JOBS)
kernel_results = rescue.candidate_result_table(
    KERNEL_WEIGHT_CONFIGS, kernel_runs, legacy_fine6_run)
display(kernel_results)

CORRECTED CLEAN MODEL SETTINGS
FINAL_REPRESENTATIONS = {'raw': 'raw_orig', 'geometry': 'geometry_white16', 'density': 'density_white5', 'coord': 'coord_white4', 'aggregates': 'aggregates_white4', 'knn': 'knn_orig', 'pca128': 'pca128_orig'}
FINAL_GAMMAS = {'raw': 0.002496, 'geometry': 0.0034171616918486107, 'density': 0.026340695648809757, 'coord': 0.02582090408371943, 'aggregates': 0.0368650778749846, 'knn': 0.04697254731412588, 'pca128': 0.0027678574834286906}


,config,weights,best_C,CV_mean,CV_std,CV_median,delta_vs_legacy_fine6,wins_vs_legacy_fine6,losses_vs_legacy_fine6,median_fold_delta,min_fold_delta,max_fold_delta,corr_with_fine6
0,K0,"{'raw': 0.68, 'geometry': 0.1, 'density': 0.1,...",3.0,0.932150,0.007132,0.932414,0.000115,13,12,1.259763e-04,-0.003543,0.004078,0.991841
1,K6,"{'raw': 0.6, 'geometry': 0.14, 'density': 0.12...",6.0,0.932079,0.007132,0.932351,0.000044,13,11,1.889645e-04,-0.003559,0.002362,0.992004
2,K1,"{'raw': 0.7, 'geometry': 0.1, 'density': 0.1, ...",3.0,0.932053,0.007224,0.932288,0.000018,14,11,1.259763e-04,-0.003937,0.004126,0.992117
3,K2,"{'raw': 0.6, 'geometry': 0.18, 'density': 0.1,...",6.0,0.932020,0.007263,0.932335,-0.000015,13,12,4.724112e-05,-0.004031,0.002331,0.992271
4,K5,"{'raw': 0.52, 'geometry': 0.2, 'density': 0.16...",6.0,0.932000,0.007173,0.932540,-0.000035,10,15,-9.448224e-05,-0.004094,0.003354,0.990583
5,K4,"{'raw': 0.58, 'geometry': 0.16, 'density': 0.1...",6.0,0.931992,0.007159,0.932303,-0.000043,13,12,1.574704e-05,-0.003842,0.002488,0.991516
6,K3,"{'raw': 0.6, 'geometry': 0.1, 'density': 0.18,...",3.0,0.931934,0.007181,0.932209,-0.000101,12,13,-7.873520e-05,-0.004456,0.005417,0.989747
7,K8,"{'raw': 0.58, 'geometry': 0.16, 'density': 0.1...",6.0,0.931804,0.007142,0.932178,-0.000231,12,13,-2.220446e-16,-0.004173,0.002331,0.991877
8,K7,"{'raw': 0.56, 'geometry': 0.2, 'density': 0.16...",6.0,0.931761,0.007104,0.931642,-0.000274,12,13,-2.362056e-04,-0.003071,0.003008,0.991128
9,K9,"{'raw': 0.55, 'geometry': 0.16, 'density': 0.1...",6.0,0.931609,0.007096,0.931941,-0.000426,11,14,-2.834467e-04,-0.004598,0.002236,0.991368


## 6. Select best kernel and calibrate one auxiliary model


In [6]:
# Strictly select the top CV result among the fixed K0-K9 candidates.
BEST_CLEAN_KERNEL = str(kernel_results.iloc[0]['config'])
best_clean_run = kernel_runs[BEST_CLEAN_KERNEL]
best_clean_row = kernel_results.iloc[0]
print(f"Legacy Fine6 CV: {legacy_fine6_run['aucs'].mean():.9f}")
print(f"Corrected best clean kernel: {BEST_CLEAN_KERNEL} "
      f"({best_clean_row['CV_mean']:.9f})")
print(f"Delta clean vs Fine6: {best_clean_row['delta_vs_legacy_fine6']:+.9f}")
print('Wins / losses:',
      int(best_clean_row['wins_vs_legacy_fine6']), '/',
      int(best_clean_row['losses_vs_legacy_fine6']))
print('Best weights:', KERNEL_WEIGHT_CONFIGS[BEST_CLEAN_KERNEL])
print('Best C:', best_clean_run['best_C'])

AUX_WEIGHT_CONFIG = {
    'clean_aux': {'pca128': 0.75, 'aggregates': 0.25},
}
C_AUX_GRID = [10.0, 30.0, 50.0, 80.0]
aux_runs = rescue.evaluate_additive_candidates(
    cache, rescue_cache, FINAL_REPRESENTATIONS, FINAL_GAMMAS,
    AUX_WEIGHT_CONFIG, C_AUX_GRID, SVC_N_JOBS)
clean_aux_run = aux_runs['clean_aux']
display(pd.DataFrame([{
    'model': 'clean_aux', 'weights': AUX_WEIGHT_CONFIG['clean_aux'],
    'best_C': clean_aux_run['best_C'],
    'CV_mean': clean_aux_run['aucs'].mean(),
    'CV_std': clean_aux_run['aucs'].std(),
}]))

Legacy Fine6 CV: 0.932035147
Corrected best clean kernel: K0 (0.932150416)
Delta clean vs Fine6: +0.000115268
Wins / losses: 13 / 12
Best weights: {'raw': 0.68, 'geometry': 0.1, 'density': 0.1, 'aggregates': 0.05, 'knn': 0.05, 'coord': 0.02, 'pca128': 0.0}
Best C: 3.0


,model,weights,best_C,CV_mean,CV_std
0,clean_aux,"{'pca128': 0.75, 'aggregates': 0.25}",30.0,0.919655,0.0074


## 7. Prediction-level fold-wise rank blending


In [7]:
BLEND_CONFIGS = {
    'CLEAN10': {'legacy_fine6': .90, 'corrected_clean_kernel': .10},
    'CLEAN20': {'legacy_fine6': .80, 'corrected_clean_kernel': .20},
    'CLEAN30': {'legacy_fine6': .70, 'corrected_clean_kernel': .30},
    'CLEAN50': {'legacy_fine6': .50, 'corrected_clean_kernel': .50},
    'CA10': {'corrected_clean_kernel': .90, 'clean_aux': .10},
    'CA15': {'corrected_clean_kernel': .85, 'clean_aux': .15},
    'CA20': {'corrected_clean_kernel': .80, 'clean_aux': .20},
    'AUX10': {'legacy_fine6': .90, 'clean_aux': .10},
    'AUX14': {'legacy_fine6': .86, 'clean_aux': .14},
    'AUX18': {'legacy_fine6': .82, 'clean_aux': .18},
    'B11_SAFE': {'legacy_fine6': .78, 'clean_aux': .22},
    'T1': {'legacy_fine6': .70, 'corrected_clean_kernel': .20,
           'clean_aux': .10},
    'T2': {'legacy_fine6': .60, 'corrected_clean_kernel': .30,
           'clean_aux': .10},
    'T3': {'legacy_fine6': .70, 'corrected_clean_kernel': .15,
           'clean_aux': .15},
    'T4': {'legacy_fine6': .60, 'corrected_clean_kernel': .25,
           'clean_aux': .15},
       'T5': {'legacy_fine6': .35, 'corrected_clean_kernel': .35,
           'clean_aux': .30},
       'T6': {'legacy_fine6': .40, 'corrected_clean_kernel': .40,
                  'clean_aux': .20},
       'T7': {'legacy_fine6': .45, 'corrected_clean_kernel': .45,
                  'clean_aux': .10},
}
blend_model_oof = {
    'legacy_fine6': legacy_fine6_run['oof'],
    'corrected_clean_kernel': best_clean_run['oof'],
    'clean_aux': clean_aux_run['oof'],
}
# run_rank_blend_configs ranks predictions independently inside every validation fold.
blend_runs = legacy.run_rank_blend_configs(cache, blend_model_oof, BLEND_CONFIGS)
blend_results = rescue.blend_result_table(cache, blend_runs, legacy_fine6_run)
display(blend_results)

BEST_BLEND = str(blend_results.iloc[0]['blend'])
CV_FINALISTS = {
    'F1': {'kind': 'single', 'config': BEST_CLEAN_KERNEL,
           'CV_mean': float(best_clean_run['aucs'].mean())},
    'F2': {'kind': 'blend', 'config': BEST_BLEND,
           'CV_mean': float(blend_runs[BEST_BLEND]['aucs'].mean())},
    'F_SAFE': {'kind': 'blend', 'config': 'B11_SAFE',
               'CV_mean': float(blend_runs['B11_SAFE']['aucs'].mean())},
}
print('Best blend:', BEST_BLEND, BLEND_CONFIGS[BEST_BLEND])
print('B11_SAFE:', BLEND_CONFIGS['B11_SAFE'])
display(pd.DataFrame([
    {'finalist': name, **spec} for name, spec in CV_FINALISTS.items()]))

,blend,CV_mean,CV_std,CV_median,delta_vs_fine6,wins_vs_fine6,losses_vs_fine6,corr_with_fine6,median_fold_delta,min_fold_delta,max_fold_delta
0,T6,0.933561,0.006495,0.933256,0.001526,20,5,0.994090,0.001724,-0.001354,0.004244
1,T5,0.933465,0.006331,0.933414,0.001430,18,7,0.991374,0.001299,-0.002071,0.005504
2,T4,0.933377,0.006657,0.932839,0.001342,21,4,0.996140,0.001228,-0.000598,0.003370
3,T7,0.933310,0.006814,0.933256,0.001275,21,4,0.995703,0.001480,-0.000850,0.003488
4,CA20,0.933263,0.006581,0.932862,0.001228,18,7,0.989722,0.001953,-0.003149,0.004575
5,T3,0.933242,0.006700,0.932280,0.001207,19,6,0.996555,0.001394,-0.000331,0.003181
6,T2,0.933238,0.006795,0.932792,0.001203,21,4,0.996666,0.001394,-0.000504,0.002708
7,B11_SAFE,0.933161,0.006531,0.933359,0.001126,19,6,0.995489,0.000976,-0.001417,0.004425
8,CA15,0.933137,0.006729,0.932847,0.001102,18,7,0.990357,0.001866,-0.003008,0.004677
9,T1,0.933121,0.006832,0.932335,0.001086,22,3,0.997113,0.001063,-0.000276,0.002386


Best blend: T6 {'legacy_fine6': 0.4, 'corrected_clean_kernel': 0.4, 'clean_aux': 0.2}
B11_SAFE: {'legacy_fine6': 0.78, 'clean_aux': 0.22}


,finalist,kind,config,CV_mean
0,F1,single,K0,0.932150
1,F2,blend,T6,0.933561
2,F_SAFE,blend,B11_SAFE,0.933161


## 8. External validation sanity check


In [8]:
X_val = val_df[features].to_numpy()
y_val = val_df.target.to_numpy()
validation_bundle = rescue.build_external_views(
    X_train, y_train, X_val,
    builder_seed=SEED + legacy.INNER_SEED_OFFSET,
    n_bins=COORD_N_BINS,
    alpha=COORD_ALPHA,
    pca_components=PCA_COMPONENTS,
)
validation_components_raw = rescue.fit_external_component_scores(
    validation_bundle,
    y_train,
    FINAL_REPRESENTATIONS,
    FINAL_GAMMAS,
    KERNEL_WEIGHT_CONFIGS[BEST_CLEAN_KERNEL],
    best_clean_run['best_C'],
    AUX_WEIGHT_CONFIG['clean_aux'],
    clean_aux_run['best_C'],
    LEGACY_FINE6_WEIGHTS,
    LEGACY_FINE6_GAMMAS,
    LEGACY_FINE6_C,
)
validation_components = {
    'legacy_fine6': validation_components_raw['legacy_fine6'],
    'corrected_clean_kernel': validation_components_raw['best_clean_kernel'],
    'clean_aux': validation_components_raw['clean_aux'],
}
fine6_validation_auc = roc_auc_score(
    y_val, validation_components['legacy_fine6'])
validation_scores = {
    'legacy_fine6': validation_components['legacy_fine6'],
    'F1': validation_components['corrected_clean_kernel'],
    'F2': rescue.external_rank_blend(
        validation_components, BLEND_CONFIGS[BEST_BLEND]),
    'B11_SAFE': rescue.external_rank_blend(
        validation_components, BLEND_CONFIGS['B11_SAFE']),
}
validation_cv = {
    'legacy_fine6': float(legacy_fine6_run['aucs'].mean()),
    'F1': CV_FINALISTS['F1']['CV_mean'],
    'F2': CV_FINALISTS['F2']['CV_mean'],
    'B11_SAFE': CV_FINALISTS['F_SAFE']['CV_mean'],
}
validation_results = pd.DataFrame([
    {
        'model': name,
        'CV_mean': validation_cv[name],
        'validation_auc': roc_auc_score(y_val, score),
        'delta_validation_vs_fine6': (
            roc_auc_score(y_val, score) - fine6_validation_auc),
    }
    for name, score in validation_scores.items()
])
display(validation_results)
# Validation is diagnostic only. It must not change models or weights.

,model,CV_mean,validation_auc,delta_validation_vs_fine6
0,legacy_fine6,0.932035,0.955038,0.000000
1,F1,0.932150,0.954069,-0.000969
2,F2,0.933561,0.955166,0.000128
3,B11_SAFE,0.933161,0.955134,0.000096


## 9. Final full-train → test fit


In [9]:
X_test = test_df[features].to_numpy()
test_bundle = rescue.build_external_views(
    X_train, y_train, X_test,
    builder_seed=SEED + legacy.INNER_SEED_OFFSET,
    n_bins=COORD_N_BINS,
    alpha=COORD_ALPHA,
    pca_components=PCA_COMPONENTS,
)
test_components_raw = rescue.fit_external_component_scores(
    test_bundle,
    y_train,
    FINAL_REPRESENTATIONS,
    FINAL_GAMMAS,
    KERNEL_WEIGHT_CONFIGS[BEST_CLEAN_KERNEL],
    best_clean_run['best_C'],
    AUX_WEIGHT_CONFIG['clean_aux'],
    clean_aux_run['best_C'],
    LEGACY_FINE6_WEIGHTS,
    LEGACY_FINE6_GAMMAS,
    LEGACY_FINE6_C,
)
test_components = {
    'legacy_fine6': test_components_raw['legacy_fine6'],
    'corrected_clean_kernel': test_components_raw['best_clean_kernel'],
    'clean_aux': test_components_raw['clean_aux'],
}

def finalist_test_score(finalist):
    if finalist == 'F1':
        raw_score = test_components['corrected_clean_kernel']
    elif finalist == 'F2':
        raw_score = rescue.external_rank_blend(
            test_components, BLEND_CONFIGS[BEST_BLEND])
    elif finalist in {'F_SAFE', 'B11_SAFE'}:
        raw_score = rescue.external_rank_blend(
            test_components, BLEND_CONFIGS['B11_SAFE'])
    else:
        raise KeyError(finalist)
    return legacy.percentile_rank(raw_score)

# The corrected submission must contain the corrected clean kernel. If the
# overall best blend is an aux-only route (including B11), retain F1 instead.
best_blend_uses_corrected = (
    BLEND_CONFIGS[BEST_BLEND].get('corrected_clean_kernel', 0.0) > 0.0)
CORRECTED_BEST_FINALIST = (
    'F2'
    if (best_blend_uses_corrected
        and CV_FINALISTS['F2']['CV_mean'] > CV_FINALISTS['F1']['CV_mean'])
    else 'F1'
)
corrected_best_test_score = finalist_test_score(CORRECTED_BEST_FINALIST)
b11_safe_test_score = finalist_test_score('F_SAFE')
print('Corrected submission finalist:', CORRECTED_BEST_FINALIST)
print('Safe submission finalist: B11_SAFE')

Corrected submission finalist: F2
Safe submission finalist: B11_SAFE


## 10. Submission files


In [10]:
def make_rescue_submission(scores, output_path):
    submission = sample_submission[['row_id']].copy()
    submission['target'] = np.asarray(scores, dtype=float)
    submission = submission[['row_id', 'target']]
    assert len(submission) == len(test_df)
    assert np.array_equal(
        submission['row_id'].to_numpy(),
        test_df['row_id'].to_numpy(),
    )
    assert np.isfinite(submission['target']).all()
    assert submission['target'].between(0, 1).all()
    submission.to_csv(output_path, index=False)
    return submission


CORRECTED_SUBMISSION_PATH = Path('submission_rescue_corrected_best.csv')
B11_SAFE_SUBMISSION_PATH = Path('submission_rescue_b11_safe.csv')
submission_rescue_corrected_best = make_rescue_submission(
    corrected_best_test_score, CORRECTED_SUBMISSION_PATH)
submission_rescue_b11_safe = make_rescue_submission(
    b11_safe_test_score, B11_SAFE_SUBMISSION_PATH)

print('Feature representation results')
display(representation_results)
print('Corrected FINAL_REPRESENTATIONS')
print(FINAL_REPRESENTATIONS)
print('Corrected FINAL_GAMMAS')
print(FINAL_GAMMAS)
print(f'Raw sanity CV: {historical_raw_CV:.9f}')
print(f"Legacy Fine6 CV: {legacy_fine6_run['aucs'].mean():.9f}")
print('K0-K9 corrected table')
display(kernel_results)
print('Best corrected clean kernel:', BEST_CLEAN_KERNEL)
print('Blend table')
display(blend_results)
print('Best blend:', BEST_BLEND)
print('B11_SAFE:', BLEND_CONFIGS['B11_SAFE'])
print('Validation table')
display(validation_results)
print('Submission files')
print(CORRECTED_SUBMISSION_PATH.resolve(), '->', CORRECTED_BEST_FINALIST)
print(B11_SAFE_SUBMISSION_PATH.resolve(), '-> B11_SAFE')
display(submission_rescue_corrected_best.head())
display(submission_rescue_b11_safe.head())

Feature representation results


,representation,source_block,dim,gamma,C,quick_CV,full25_CV,full25_std,delta_vs_orig,wins_vs_orig,losses_vs_orig
0,geometry_orig,geometry,32,0.003193,10.0,0.916078,0.914880,0.011394,0.000000,0,0
1,geometry_white16,geometry,16,0.003417,10.0,0.918377,0.917002,0.011952,0.002122,18,7
2,geometry_family,geometry,14,0.014711,10.0,0.916909,0.915306,0.010947,0.000426,14,11
3,density_orig,density,14,0.005985,10.0,0.899811,0.901171,0.013576,0.000000,0,0
4,density_white5,density,5,0.026341,6.0,0.904907,0.905047,0.010986,0.003876,21,4
5,density_family,density,6,0.141479,6.0,0.906459,0.904699,0.011059,0.003528,19,6
6,coord_orig,coord,10,0.013867,3.0,0.874704,0.874701,0.013711,0.000000,0,0
7,coord_white4,coord,4,0.025821,3.0,0.879847,0.877664,0.012985,0.002963,22,3
8,aggregates_orig,aggregates,6,0.018459,1.0,0.836719,0.837635,0.016542,0.000000,0,0
9,aggregates_white4,aggregates,4,0.036865,6.0,0.839969,0.839761,0.016255,0.002126,17,8


Corrected FINAL_REPRESENTATIONS
{'raw': 'raw_orig', 'geometry': 'geometry_white16', 'density': 'density_white5', 'coord': 'coord_white4', 'aggregates': 'aggregates_white4', 'knn': 'knn_orig', 'pca128': 'pca128_orig'}
Corrected FINAL_GAMMAS
{'raw': 0.002496, 'geometry': 0.0034171616918486107, 'density': 0.026340695648809757, 'coord': 0.02582090408371943, 'aggregates': 0.0368650778749846, 'knn': 0.04697254731412588, 'pca128': 0.0027678574834286906}
Raw sanity CV: 0.929060217
Legacy Fine6 CV: 0.932035147
K0-K9 corrected table


,config,weights,best_C,CV_mean,CV_std,CV_median,delta_vs_legacy_fine6,wins_vs_legacy_fine6,losses_vs_legacy_fine6,median_fold_delta,min_fold_delta,max_fold_delta,corr_with_fine6
0,K0,"{'raw': 0.68, 'geometry': 0.1, 'density': 0.1,...",3.0,0.932150,0.007132,0.932414,0.000115,13,12,1.259763e-04,-0.003543,0.004078,0.991841
1,K6,"{'raw': 0.6, 'geometry': 0.14, 'density': 0.12...",6.0,0.932079,0.007132,0.932351,0.000044,13,11,1.889645e-04,-0.003559,0.002362,0.992004
2,K1,"{'raw': 0.7, 'geometry': 0.1, 'density': 0.1, ...",3.0,0.932053,0.007224,0.932288,0.000018,14,11,1.259763e-04,-0.003937,0.004126,0.992117
3,K2,"{'raw': 0.6, 'geometry': 0.18, 'density': 0.1,...",6.0,0.932020,0.007263,0.932335,-0.000015,13,12,4.724112e-05,-0.004031,0.002331,0.992271
4,K5,"{'raw': 0.52, 'geometry': 0.2, 'density': 0.16...",6.0,0.932000,0.007173,0.932540,-0.000035,10,15,-9.448224e-05,-0.004094,0.003354,0.990583
5,K4,"{'raw': 0.58, 'geometry': 0.16, 'density': 0.1...",6.0,0.931992,0.007159,0.932303,-0.000043,13,12,1.574704e-05,-0.003842,0.002488,0.991516
6,K3,"{'raw': 0.6, 'geometry': 0.1, 'density': 0.18,...",3.0,0.931934,0.007181,0.932209,-0.000101,12,13,-7.873520e-05,-0.004456,0.005417,0.989747
7,K8,"{'raw': 0.58, 'geometry': 0.16, 'density': 0.1...",6.0,0.931804,0.007142,0.932178,-0.000231,12,13,-2.220446e-16,-0.004173,0.002331,0.991877
8,K7,"{'raw': 0.56, 'geometry': 0.2, 'density': 0.16...",6.0,0.931761,0.007104,0.931642,-0.000274,12,13,-2.362056e-04,-0.003071,0.003008,0.991128
9,K9,"{'raw': 0.55, 'geometry': 0.16, 'density': 0.1...",6.0,0.931609,0.007096,0.931941,-0.000426,11,14,-2.834467e-04,-0.004598,0.002236,0.991368


Best corrected clean kernel: K0
Blend table


...

Best blend: T6
B11_SAFE: {'legacy_fine6': 0.78, 'clean_aux': 0.22}
Validation table


,model,CV_mean,validation_auc,delta_validation_vs_fine6
0,legacy_fine6,0.932035,0.955038,0.000000
1,F1,0.932150,0.954069,-0.000969
2,F2,0.933561,0.955166,0.000128
3,B11_SAFE,0.933161,0.955134,0.000096


Submission files
C:\Users\peret\Desktop\Third year\ML\comp\comp 1\sbmt_kaggle\submission_rescue_corrected_best.csv -> F2
C:\Users\peret\Desktop\Third year\ML\comp\comp 1\sbmt_kaggle\submission_rescue_b11_safe.csv -> B11_SAFE


,row_id,target
0,row_46ad1afad8562239f79c,0.831541
1,row_b42ade9cd57b8d919acb,0.758961
2,row_c951cd5e197b6b4e8b83,0.233423
3,row_a54b3423760d599416ce,0.040323
4,row_a87c87cc47690e2a303f,0.896953


,row_id,target
0,row_46ad1afad8562239f79c,0.831541
1,row_b42ade9cd57b8d919acb,0.768817
2,row_c951cd5e197b6b4e8b83,0.189964
3,row_a54b3423760d599416ce,0.048387
4,row_a87c87cc47690e2a303f,0.883513
